# Panel Cointegration Tests: Long-Run Relationships in Non-Stationary Data

**Version**: 1.0  
**Date**: 2026-02-22  
**Estimated Duration**: 110 minutes  
**Difficulty**: Intermediate-Advanced

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Define** cointegration and explain its economic interpretation
2. **Distinguish** between spurious regression and cointegrating regression
3. **Apply** three major panel cointegration tests: Pedroni, Westerlund, Kao
4. **Interpret** the difference between panel statistics and group statistics
5. **Extract** and analyze cointegration vectors (β coefficients)
6. **Recognize** when to use error correction models (ECM)
7. **Compare** results across different cointegration testing methods
8. **Apply** cointegration tests to real economic data (PPP, consumption-income)

## Prerequisites

- Completion of Notebook 01 (Unit Root Tests)
- Understanding of I(0) vs I(1) series
- Familiarity with regression residuals

## Datasets

| Dataset | Description | N | T | Variables |
|---------|-------------|---|---|-----------|
| `oecd_macro.csv` | OECD consumption-income | 20 countries | 40 years | consumption, income, investment |
| `ppp_data.csv` | PPP exchange rates | 25 countries | 35 years | exchange_rate, prices |
| `interest_rates.csv` | Interest rate parity | 15 countries | 30 years | domestic_rate, us_rate, spread |

## Estimated Time per Section

| Section | Topic | Time |
|---------|-------|------|
| 1 | What is Cointegration? | 15 min |
| 2 | Panel Cointegration Tests | 25 min |
| 3 | Comparison and Applications | 30 min |
| 4 | Error Correction Models | 15 min |
| 5 | Key Takeaways | 5 min |
| Exercises | Practice | 20 min |

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# Local utilities
import sys

import statsmodels.api as sm

# PanelBox imports
import panelbox
from panelbox.datasets import load_dataset
from panelbox.diagnostics.cointegration import kao_test, pedroni_test, westerlund_test
from panelbox.diagnostics.unit_root import panel_unit_root_test

BASE_DIR = Path("..")
sys.path.insert(0, str(BASE_DIR / "utils"))
from cointegration_helpers import (
    compare_cointegration_methods,
    compute_half_lives,
)

# Configuration
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 11
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)
np.random.seed(42)

# Paths
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures" / "cointegration"
TABLES_DIR = OUTPUT_DIR / "tables" / "cointegration"
RESULTS_DIR = OUTPUT_DIR / "results" / "cointegration"

for d in [FIGURES_DIR, TABLES_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete!")
print(f"PanelBox version: {panelbox.__version__}")
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

---

## Section 1: What is Cointegration?

### 1.1 Informal and Formal Definition

**Informal definition:**  
Two variables "move together" in the long run, even though both wander randomly in the short run.

**Formal definition** (Engle & Granger 1987):  
Variables $y_t$ and $x_t$ are **cointegrated** if:
1. Both are I(1) (non-stationary with unit roots)
2. There exists $\beta$ such that: $z_t = y_t - \beta \cdot x_t$ is I(0) (stationary)

**Notation:**
- We write: $y_t \sim I(1)$, $x_t \sim I(1)$, but $(y_t - \beta \cdot x_t) \sim I(0)$
- $\beta$ is called the **cointegration coefficient** or **cointegration vector**

### Key Insight

Cointegration implies a **long-run equilibrium relationship**. Deviations from this equilibrium are temporary -- the system self-corrects.

In [ ]:
# Visual demonstration: two cointegrated series
np.random.seed(42)
T = 200

# True relationship: y_t = 2*x_t + u_t, where u_t is stationary
x = np.cumsum(np.random.randn(T))  # I(1) random walk
u = 0.5 * np.random.randn(T)  # I(0) stationary error
y = 2 * x + u  # Cointegrated with x!

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Panel 1: Both series (I(1))
axes[0].plot(x, label="$x_t$ (I(1))", linewidth=1.2)
axes[0].plot(y, label="$y_t$ (I(1))", linewidth=1.2)
axes[0].set_title("Two Non-Stationary Series", fontweight="bold")
axes[0].legend()
axes[0].set_xlabel("Time")

# Panel 2: Cointegration residual (I(0))
z = y - 2 * x
axes[1].plot(z, color="green", linewidth=1.0)
axes[1].axhline(0, color="red", linestyle="--", alpha=0.7)
axes[1].set_title("$z_t = y_t - 2 x_t$ (I(0))", fontweight="bold")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Residual")

# Panel 3: Scatter with cointegration line
axes[2].scatter(x, y, alpha=0.4, s=15)
x_line = np.array([x.min(), x.max()])
axes[2].plot(x_line, 2 * x_line, "r--", linewidth=2, label=r"$y = 2x$ (true)")
axes[2].set_title("Cointegrating Relationship", fontweight="bold")
axes[2].set_xlabel("$x_t$")
axes[2].set_ylabel("$y_t$")
axes[2].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_cointegration_concept.png", dpi=300, bbox_inches="tight")
plt.show()

print("Key observations:")
print(f"  x_t std dev: {x.std():.2f} (growing with T -- non-stationary)")
print(f"  y_t std dev: {y.std():.2f} (growing with T -- non-stationary)")
print(f"  z_t = y - 2x std dev: {z.std():.2f} (bounded -- stationary!)")

### 1.2 Economic Interpretation

Cointegration has natural economic interpretations:

| Example | Variables | Theory | Cointegration Meaning |
|---------|-----------|--------|-----------------------|
| **Purchasing Power Parity** | Exchange rate, prices | $S_t \approx P_t / P^*_t$ | PPP deviations are temporary |
| **Consumption-Income** | $C_t$, $Y_t$ | Permanent Income Hypothesis | $C/Y$ ratio stable long-term |
| **Interest Rate Parity** | $i_t$, $i^*_t$, forward premium | Covered IRP | Interest differentials are temporary |

**Why cointegration matters:**
- **Modeling**: Can use levels (don't need to difference)
- **Interpretation**: Coefficients have long-run meaning
- **Forecasting**: Long-run forecasts are improved
- **Policy**: Shocks have temporary effects; system returns to equilibrium

### 1.3 Cointegration vs. Spurious Regression

**Spurious regression** (Granger & Newbold 1974):
- Regressing one I(1) series on another **independent** I(1) series
- High $R^2$, significant t-stats -- but no real relationship!
- Residuals are I(1) $\rightarrow$ invalid inference

**Cointegrating regression**:
- Regressing one I(1) series on a **related** I(1) series
- High $R^2$, significant t-stats -- and a **real** relationship
- Residuals are I(0) $\rightarrow$ valid inference (with correction)

**How to tell them apart?** Test whether the regression residuals are stationary. That is exactly what cointegration tests do.

In [ ]:
# Demonstration: spurious regression vs. cointegrating regression
np.random.seed(42)
n = 200

# Spurious: two independent random walks
x_spur = np.cumsum(np.random.randn(n))
y_spur = np.cumsum(np.random.randn(n))

# Cointegrating: y = 1.5*x + stationary error
x_coint = np.cumsum(np.random.randn(n))
y_coint = 1.5 * x_coint + np.random.randn(n) * 0.5

# Run OLS on both
model_spur = sm.OLS(y_spur, sm.add_constant(x_spur)).fit()
model_coint = sm.OLS(y_coint, sm.add_constant(x_coint)).fit()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Top row: series plots
axes[0, 0].plot(x_spur, label="x (RW)", alpha=0.8)
axes[0, 0].plot(y_spur, label="y (RW)", alpha=0.8)
axes[0, 0].set_title(f"Spurious ($R^2$={model_spur.rsquared:.3f})", fontweight="bold")
axes[0, 0].legend()

axes[0, 1].plot(x_coint, label="x (RW)", alpha=0.8)
axes[0, 1].plot(y_coint, label="y = 1.5x + noise", alpha=0.8)
axes[0, 1].set_title(f"Cointegrated ($R^2$={model_coint.rsquared:.3f})", fontweight="bold")
axes[0, 1].legend()

# Bottom row: residuals
axes[1, 0].plot(model_spur.resid, color="red", linewidth=0.8)
axes[1, 0].axhline(0, color="black", linestyle="--", alpha=0.5)
axes[1, 0].set_title("Residuals: I(1) -- Wandering!", fontweight="bold")
axes[1, 0].set_xlabel("Time")

axes[1, 1].plot(model_coint.resid, color="green", linewidth=0.8)
axes[1, 1].axhline(0, color="black", linestyle="--", alpha=0.5)
axes[1, 1].set_title("Residuals: I(0) -- Mean-Reverting!", fontweight="bold")
axes[1, 1].set_xlabel("Time")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_spurious_vs_cointegrated.png", dpi=300, bbox_inches="tight")
plt.show()

print("Spurious Regression:")
print(f"  R-squared: {model_spur.rsquared:.4f}")
print(f"  t-stat on x: {model_spur.tvalues[1]:.4f} (p={model_spur.pvalues[1]:.4f})")
print(f"  DW statistic: {sm.stats.durbin_watson(model_spur.resid):.4f}")
print(
    f"  -> R^2 > DW: {model_spur.rsquared > sm.stats.durbin_watson(model_spur.resid)} (Granger-Newbold rule)"
)
print()
print("Cointegrating Regression:")
print(f"  R-squared: {model_coint.rsquared:.4f}")
print(f"  t-stat on x: {model_coint.tvalues[1]:.4f} (p={model_coint.pvalues[1]:.4f})")
print(f"  DW statistic: {sm.stats.durbin_watson(model_coint.resid):.4f}")
print(f"  -> R^2 > DW: {model_coint.rsquared > sm.stats.durbin_watson(model_coint.resid)}")

### 1.4 Panel Cointegration Advantages

**Why panel cointegration tests?**

1. **More power**: $N \times T$ observations vs $T$ for time series
2. **Heterogeneity**: Allow $\beta_i$ to vary across entities
3. **Shorter T**: Can detect cointegration with $T=20$-$30$ (vs $T=100$+ for time series)
4. **Cross-validation**: $N$ entities provide robustness

---

## Section 2: Panel Cointegration Tests

PanelBox implements three major panel cointegration tests:

| Method | # Tests | Heterogeneity | Bootstrap | Complexity | When to Use |
|--------|---------|---------------|-----------|------------|-------------|
| **Pedroni** (1999, 2004) | 7 | $\beta_i$ varies | No | Medium | Default choice; comprehensive |
| **Westerlund** (2007) | 4 | $\beta_i$, $\gamma_i$ vary | Yes | High | Cross-sectional dependence |
| **Kao** (1999) | 1-2 | $\beta$ homogeneous | No | Low | Quick check; homogeneous panels |

All three tests share the same **null hypothesis**:

> **H0**: No cointegration (residuals have unit root)

### 2.1 Pedroni (1999, 2004) Tests

**Most comprehensive**: 7 different test statistics in two categories.

**Cointegrating regression** (run for each entity $i$):
$$y_{it} = \alpha_i + \beta_i x_{it} + \varepsilon_{it}$$

Then test whether the residuals $\hat{\varepsilon}_{it}$ are I(0).

**Panel statistics** (within-dimension): Pool across entities, assume some homogeneity
- panel_v, panel_rho, panel_PP, panel_ADF

**Group statistics** (between-dimension): Allow full heterogeneity
- group_rho, group_PP, group_ADF

**Interpretation guidelines:**
- 0-2 rejections: Weak/no evidence of cointegration
- 3-4 rejections: Moderate evidence
- 5-7 rejections: Strong evidence of cointegration

In [ ]:
# Load OECD data for demonstration
oecd = load_dataset("oecd_macro", category="diagnostics")

print("OECD Macro Data")
print("=" * 60)
print(f"Shape: {oecd.shape}")
print(f"Countries: {oecd['country'].nunique()}")
print(f"Years: {oecd['year'].min()} - {oecd['year'].max()}")
print(f"\nColumns: {list(oecd.columns)}")
print("\nFirst rows:")
display(oecd.head())
print("\nSummary statistics:")
display(oecd.describe())

In [ ]:
# Step 0: Verify variables are I(1) before testing cointegration
print("Step 0: Unit Root Tests (verify I(1))")
print("=" * 60)
print("Variables must be I(1) before testing for cointegration.\n")

for var_name in ["log_C", "log_Y"]:
    print(f"Testing: {var_name}")
    print("-" * 40)
    try:
        ur_result = panel_unit_root_test(
            oecd, var_name, entity_col="country", time_col="year", test="all", trend="ct"
        )
        print(ur_result.summary_table())
    except Exception as e:
        print(f"  Error: {e}")
    print()

In [ ]:
# Pedroni test: consumption-income cointegration
print("Pedroni (1999, 2004) Test: log(C) ~ log(Y)")
print("=" * 60)
print("H0: No cointegration (residuals have unit root)")
print("H1: Cointegration (residuals are stationary)\n")

try:
    pedroni_result = pedroni_test(
        data=oecd,
        entity_col="country",
        time_col="year",
        y_var="log_C",
        x_vars=["log_Y"],
        method="all",
        trend="c",
    )

    print(pedroni_result.summary())

    # Count rejections
    rejections = pedroni_result.reject_at(alpha=0.05)
    if isinstance(rejections, dict):
        n_reject = sum(1 for v in rejections.values() if v)
        n_total = len(rejections)
    else:
        n_reject = 1 if rejections else 0
        n_total = 1

    print(f"\nRejections at 5%: {n_reject} / {n_total}")
    if n_reject >= 5:
        print("-> Strong evidence of cointegration")
    elif n_reject >= 3:
        print("-> Moderate evidence of cointegration")
    else:
        print("-> Weak/no evidence of cointegration")

except Exception as e:
    print(f"Pedroni test error: {e}")

### 2.2 Westerlund (2007) Error Correction Tests

**Key innovation:** Tests error correction directly (no need to estimate the cointegrating regression first).

**Error correction model:**
$$\Delta y_{it} = \gamma_i(y_{i,t-1} - \beta_i x_{i,t-1}) + \sum_j \theta_{ij} \Delta y_{i,t-j} + \sum_j \delta_{ij} \Delta x_{i,t-j} + \varepsilon_{it}$$

**H0:** $\gamma_i = 0$ for all $i$ (no error correction $\rightarrow$ no cointegration)  
**H1:** $\gamma_i < 0$ for at least some $i$ (error correction $\rightarrow$ cointegration)

**Interpretation of $\gamma_i$:**
- $\gamma_i = -0.2$ means 20% of deviation from equilibrium is corrected each period
- Larger $|\gamma_i|$ means faster adjustment

**Four test statistics:**
- **Gt, Ga**: Group mean tests (allow heterogeneity; H1: $\gamma_i < 0$ for at least one $i$)
- **Pt, Pa**: Panel tests (assume some homogeneity; pool across entities)

**Bootstrap:** Standard critical values assume cross-sectional independence. Use bootstrap for robust p-values when dependence is a concern.

In [ ]:
# Westerlund test (without bootstrap for speed)
print("Westerlund (2007) ECM Test: log(C) ~ log(Y)")
print("=" * 60)
print("H0: No error correction (gamma_i = 0) -> no cointegration")
print("H1: Error correction (gamma_i < 0) -> cointegration\n")

try:
    westerlund_result = westerlund_test(
        data=oecd,
        entity_col="country",
        time_col="year",
        y_var="log_C",
        x_vars=["log_Y"],
        method="all",
        trend="c",
        use_bootstrap=False,  # Skip bootstrap for speed
    )

    print(westerlund_result.summary())

    # Count rejections
    rejections = westerlund_result.reject_at(alpha=0.05)
    if isinstance(rejections, dict):
        n_reject = sum(1 for v in rejections.values() if v)
        n_total = len(rejections)
    else:
        n_reject = 1 if rejections else 0
        n_total = 1

    print(f"\nRejections at 5%: {n_reject} / {n_total}")

except Exception as e:
    print(f"Westerlund test error: {e}")

In [ ]:
# Westerlund with bootstrap (may take a minute)
print("Westerlund with Bootstrap (n=199 for demonstration)")
print("=" * 60)
print("NOTE: Bootstrap provides robust p-values under cross-sectional dependence.")
print("      Using n_bootstrap=199 for speed; use 999+ for final results.\n")

try:
    westerlund_boot = westerlund_test(
        data=oecd,
        entity_col="country",
        time_col="year",
        y_var="log_C",
        x_vars=["log_Y"],
        method="all",
        trend="c",
        use_bootstrap=True,
        n_bootstrap=199,
        random_state=42,
    )

    print(westerlund_boot.summary())

except Exception as e:
    print(f"Westerlund bootstrap error: {e}")
    print("This is expected if bootstrap is computationally intensive.")
    print("The non-bootstrap results above are still valid under independence.")

### 2.3 Kao (1999) Test

**Simplest approach:** Extends Dickey-Fuller to panels.

**Assumptions:**
- Homogeneous cointegration coefficient: $\beta_i = \beta$ for all $i$
- Simpler than Pedroni (single $\beta$)
- Fastest computation

**Procedure:**
1. Pool all data and estimate: $y_{it} = \alpha_i + \beta x_{it} + \varepsilon_{it}$
2. Extract pooled residuals $\hat{\varepsilon}_{it}$
3. Test H0: residuals have unit root

**When to use Kao:**
- Homogeneity is plausible (e.g., similar countries)
- Large N, moderate T (computationally efficient)
- Quick diagnostic before more detailed tests

In [ ]:
# Kao test
print("Kao (1999) Test: log(C) ~ log(Y)")
print("=" * 60)
print("H0: No cointegration (residuals have unit root)")
print("Assumes: Homogeneous beta across all countries\n")

try:
    kao_result = kao_test(
        data=oecd,
        entity_col="country",
        time_col="year",
        y_var="log_C",
        x_vars=["log_Y"],
        method="all",
    )

    print(kao_result.summary())

    # Check rejection
    rejections = kao_result.reject_at(alpha=0.05)
    if isinstance(rejections, dict):
        for test_name, reject in rejections.items():
            print(f"  {test_name}: {'REJECT H0' if reject else 'Fail to reject H0'}")
    else:
        print(f"  Reject H0: {rejections}")

except Exception as e:
    print(f"Kao test error: {e}")

---

## Section 3: Comparison and Applications

### 3.1 Comparing All Three Methods

Let us use the utility function `compare_cointegration_methods()` to run all three tests simultaneously and compare.

In [ ]:
# Compare all three methods on consumption-income
print("Comparison: All Three Cointegration Tests")
print("=" * 60)
print("Variable: log(C) ~ log(Y)\n")

comparison_df = compare_cointegration_methods(
    data=oecd,
    y_var="log_C",
    x_vars=["log_Y"],
    entity_col="country",
    time_col="year",
    save_path=str(TABLES_DIR / "02_cointegration_comparison.csv"),
)

display(comparison_df)

# Summarize
if "Reject_5pct" in comparison_df.columns:
    n_reject = comparison_df["Reject_5pct"].sum()
    n_total = len(comparison_df)
    print(f"\nTotal rejections: {n_reject} / {n_total}")

    # By method
    for method in comparison_df["Method"].unique():
        subset = comparison_df[comparison_df["Method"] == method]
        n_r = subset["Reject_5pct"].sum()
        n_t = len(subset)
        print(f"  {method}: {n_r} / {n_t} rejections")

### 3.2 Practical Application 1: Purchasing Power Parity (PPP)

**Economic theory:** The exchange rate should equal the ratio of price levels:
$$S_t = P_t / P^*_t \quad \Rightarrow \quad \log(S_t) = \log(P_t) - \log(P^*_t)$$

**Cointegration implication:** If PPP holds in the long run, then $\log(S)$, $\log(P)$, $\log(P^*)$ are cointegrated.

In [ ]:
# Load PPP data
ppp = load_dataset("ppp_data", category="diagnostics")

print("PPP Data")
print("=" * 60)
print(f"Shape: {ppp.shape}")
print(f"Countries: {ppp['country'].nunique()} ({sorted(ppp['country'].unique())})")
print(f"Years: {ppp['year'].min()} - {ppp['year'].max()}")
print(f"\nColumns: {list(ppp.columns)}")
display(ppp.head())

In [ ]:
# Create log variables for PPP test
ppp["log_P"] = np.log(ppp["price_domestic"])
ppp["log_Pstar"] = np.log(ppp["price_foreign"])

# Visualize: exchange rate vs price ratio
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
sample_countries = sorted(ppp["country"].unique())[:6]

for i, country in enumerate(sample_countries):
    cdata = ppp[ppp["country"] == country].sort_values("year")
    ax = axes[i]
    ax.plot(cdata["year"], cdata["log_S"], label="log(S)", linewidth=1.5)
    ax.plot(cdata["year"], cdata["log_P_ratio"], label="log(P/P*)", linewidth=1.5, linestyle="--")
    ax.set_title(country, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle("PPP: Exchange Rate vs. Price Ratio", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_ppp_series.png", dpi=300, bbox_inches="tight")
plt.show()

print("If PPP holds, log(S) and log(P/P*) should be cointegrated.")

In [ ]:
# Test PPP cointegration using all three methods
print("PPP Cointegration Tests: log(S) ~ log(P) + log(P*)")
print("=" * 60)

ppp_comparison = compare_cointegration_methods(
    data=ppp,
    y_var="log_S",
    x_vars=["log_P", "log_Pstar"],
    entity_col="country",
    time_col="year",
    save_path=str(TABLES_DIR / "02_ppp_cointegration.csv"),
)

display(ppp_comparison)

if "Reject_5pct" in ppp_comparison.columns:
    n_reject = ppp_comparison["Reject_5pct"].sum()
    n_total = len(ppp_comparison)
    print(f"\nTotal rejections: {n_reject} / {n_total}")

    if n_reject >= n_total * 0.5:
        print("Evidence supports long-run PPP")
    else:
        print("Weak evidence for PPP")

### 3.3 Practical Application 2: Consumption-Income Relationship

**Economic theory:** Permanent Income Hypothesis
- Consumption proportional to permanent income
- Transitory income doesn't affect consumption much

**Implication:** $C_t$ and $Y_t$ should be cointegrated with $\beta < 1$.

In [ ]:
# Visualize consumption-income relationship
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
sample_countries = sorted(oecd["country"].unique())[:6]

for i, country in enumerate(sample_countries):
    cdata = oecd[oecd["country"] == country].sort_values("year")
    ax = axes[i]

    # Time series plot
    ax2 = ax.twinx()
    (l1,) = ax.plot(cdata["year"], cdata["log_C"], "b-", linewidth=1.2, label="log(C)")
    (l2,) = ax2.plot(cdata["year"], cdata["log_Y"], "r--", linewidth=1.2, label="log(Y)")
    ax.set_title(country, fontweight="bold")
    ax.set_ylabel("log(C)", color="blue", fontsize=9)
    ax2.set_ylabel("log(Y)", color="red", fontsize=9)
    ax.legend([l1, l2], ["log(C)", "log(Y)"], fontsize=8)

plt.suptitle("Consumption and Income Move Together (Cointegration)", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_consumption_income_series.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Detailed Pedroni test on consumption-income
print("Pedroni Test: Consumption-Income Cointegration")
print("=" * 60)

try:
    pedroni_cy = pedroni_test(
        data=oecd,
        entity_col="country",
        time_col="year",
        y_var="log_C",
        x_vars=["log_Y"],
        method="all",
        trend="c",
    )

    print(pedroni_cy.summary())
    print(f"\nN entities: {pedroni_cy.n_entities}")
    print(f"Avg T: {pedroni_cy.n_time}")

except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Scatter plot: consumption vs income by country
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
sample_countries = sorted(oecd["country"].unique())[:6]

for i, country in enumerate(sample_countries):
    cdata = oecd[oecd["country"] == country].sort_values("year")
    ax = axes[i]

    # Scatter plot
    ax.scatter(cdata["log_Y"], cdata["log_C"], alpha=0.6, s=20)

    # Fit OLS line for this country
    X_c = sm.add_constant(cdata["log_Y"].values)
    ols_c = sm.OLS(cdata["log_C"].values, X_c).fit()
    x_range = np.array([cdata["log_Y"].min(), cdata["log_Y"].max()])
    y_pred = ols_c.params[0] + ols_c.params[1] * x_range
    ax.plot(x_range, y_pred, "r--", linewidth=2)

    ax.set_title(f"{country}\n$\\beta$={ols_c.params[1]:.3f}", fontweight="bold")
    ax.set_xlabel("log(Income)")
    ax.set_ylabel("log(Consumption)")

plt.suptitle(
    "Long-run Consumption-Income Relationship (OLS per country)", fontweight="bold", fontsize=14
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_consumption_income_scatter.png", dpi=300, bbox_inches="tight")
plt.show()

---

## Section 4: Error Correction Models

### 4.1 From Cointegration to ECM

**Granger Representation Theorem:** If variables are cointegrated, there exists an error correction representation:

$$\Delta y_t = \gamma(y_{t-1} - \beta x_{t-1}) + \sum_j \theta_j \Delta y_{t-j} + \sum_j \delta_j \Delta x_{t-j} + \varepsilon_t$$

| Term | Name | Interpretation |
|------|------|----------------|
| $\gamma(y_{t-1} - \beta x_{t-1})$ | Error correction term (ECT) | Speed of adjustment to equilibrium |
| $\sum \theta_j \Delta y_{t-j}$ | Short-run dynamics of y | Temporary effects |
| $\sum \delta_j \Delta x_{t-j}$ | Short-run dynamics of x | Temporary effects |

**Interpretation of $\gamma$:**
- $\gamma = -0.2$: 20% of deviation corrected per period
- Half-life $= -\ln(2) / \ln(1 + \gamma)$
- **Must be negative** for stable adjustment

In [ ]:
# Two-step ECM estimation (Engle-Granger approach)
print("Error Correction Model: Two-Step Engle-Granger Approach")
print("=" * 60)

# Step 1: Estimate cointegrating regression (OLS per country)
print("\nStep 1: Estimate cointegrating regression per country")
print("-" * 40)

ecm_data = oecd.copy()
ecm_data["residual"] = np.nan
betas = {}

for country in oecd["country"].unique():
    mask = ecm_data["country"] == country
    cdata = ecm_data.loc[mask].sort_values("year")
    X = sm.add_constant(cdata["log_Y"].values)
    ols = sm.OLS(cdata["log_C"].values, X).fit()
    ecm_data.loc[mask, "residual"] = ols.resid
    betas[country] = ols.params[1]

avg_beta = np.mean(list(betas.values()))
print(f"Average cointegrating coefficient (MPC): {avg_beta:.4f}")
print(f"Range: [{min(betas.values()):.4f}, {max(betas.values()):.4f}]")

# Step 2: Estimate ECM using lagged residuals
print("\nStep 2: Estimate Error Correction Model")
print("-" * 40)

# Create differences and lags
ecm_data = ecm_data.sort_values(["country", "year"])
ecm_data["d_log_C"] = ecm_data.groupby("country")["log_C"].diff()
ecm_data["d_log_Y"] = ecm_data.groupby("country")["log_Y"].diff()
ecm_data["ect_lag"] = ecm_data.groupby("country")["residual"].shift(1)

# Drop missing values
ecm_clean = ecm_data.dropna(subset=["d_log_C", "d_log_Y", "ect_lag"]).copy()

# Pooled ECM: d_log_C = gamma * ect_lag + delta * d_log_Y + epsilon
X_ecm = sm.add_constant(ecm_clean[["ect_lag", "d_log_Y"]].values)
ecm_ols = sm.OLS(ecm_clean["d_log_C"].values, X_ecm).fit()

print("\nPooled ECM Results:")
print(f"  Error correction speed (gamma): {ecm_ols.params[1]:.4f} (t={ecm_ols.tvalues[1]:.2f})")
print(f"  Short-run income effect (delta): {ecm_ols.params[2]:.4f} (t={ecm_ols.tvalues[2]:.2f})")

gamma = ecm_ols.params[1]
if gamma < 0:
    half_life = -np.log(2) / np.log(1 + gamma)
    print(f"\n  Half-life of deviations: {half_life:.2f} years")
    print(f"  Interpretation: {abs(gamma) * 100:.1f}% of disequilibrium corrected each year")
else:
    print(f"\n  WARNING: gamma >= 0 ({gamma:.4f}), no error correction detected")

In [ ]:
# Visualize ECM: cointegration residuals and half-lives
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: cointegration residuals for selected countries
ax = axes[0]
for country in sorted(oecd["country"].unique())[:5]:
    cdata = ecm_data[ecm_data["country"] == country].sort_values("year")
    ax.plot(cdata["year"], cdata["residual"], label=country, linewidth=1.0, alpha=0.8)
ax.axhline(0, color="red", linestyle="--", linewidth=1.0)
ax.set_title("Cointegration Residuals (should be stationary)", fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Residual")
ax.legend(fontsize=8)

# Right: distribution of country-specific betas
ax = axes[1]
beta_values = list(betas.values())
ax.hist(beta_values, bins=10, edgecolor="black", alpha=0.7)
ax.axvline(avg_beta, color="red", linestyle="--", linewidth=2, label=f"Mean={avg_beta:.3f}")
ax.axvline(1.0, color="green", linestyle=":", linewidth=2, label="$\\beta=1$ (unit elasticity)")
ax.set_title("Distribution of Country-Specific $\\beta_i$ (MPC)", fontweight="bold")
ax.set_xlabel("$\\beta_i$")
ax.set_ylabel("Count")
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_ecm_residuals_betas.png", dpi=300, bbox_inches="tight")
plt.show()

print("Country-specific cointegrating coefficients (MPC):")
beta_df = pd.DataFrame([{"Country": k, "Beta (MPC)": v} for k, v in sorted(betas.items())])
display(beta_df)
beta_df.to_csv(TABLES_DIR / "02_country_betas.csv", index=False)

In [ ]:
# Half-life demonstration
print("Half-Life Interpretation")
print("=" * 60)
print("How quickly does the system return to equilibrium?\n")

example_gammas = np.array([-0.05, -0.10, -0.20, -0.30, -0.50, -0.80])
hl_df = compute_half_lives(example_gammas)
hl_df.index = [f"gamma={g:.2f}" for g in example_gammas]
display(hl_df)

print(f"\nOur estimated gamma: {gamma:.4f}")
if gamma < 0:
    our_hl = compute_half_lives([gamma])
    print(
        f"Half-life: {our_hl.iloc[0]['half_life']:.2f} years ({our_hl.iloc[0]['interpretation']})"
    )

---

## Section 5: Key Takeaways

### Conceptual
1. **Cointegration** = long-run equilibrium between I(1) variables
2. **Economic interpretation**: Deviations are temporary; system returns to equilibrium
3. **Distinguishes** spurious regression from true long-run relationship
4. **Error correction** models combine short-run dynamics and long-run equilibrium
5. **Panel tests** are more powerful than time-series Engle-Granger

### Methodological
6. **Pedroni**: Most comprehensive (7 tests), default choice
7. **Westerlund**: Use when cross-sectional dependence is a concern
8. **Kao**: Use when homogeneity is plausible and efficiency is important
9. **Panel vs Group stats**: Group allows more heterogeneity
10. **Bootstrap**: Necessary for robust inference with cross-sectional dependence

### Practical
11. Always verify variables are I(1) before testing cointegration
12. Multiple tests provide robustness (5+ rejections = strong evidence)
13. Extract and interpret cointegration vectors ($\beta_i$)
14. Use ECM to estimate adjustment speeds ($\gamma$)
15. Heterogeneity is common -- allow for it unless theory dictates otherwise

---

## Troubleshooting

### Common Errors and Solutions

**1. "Variables are not I(1)" warning**
- Cointegration tests require I(1) variables. If variables are I(0), use standard regression instead.
- Solution: Run unit root tests first (Notebook 01).

**2. Bootstrap is very slow**
- Start with `n_bootstrap=99` for testing, use `999+` for final results.
- Kao is fastest for initial exploration.
- Save intermediate results to avoid re-running.

**3. Contradictory results across tests**
- This is common. Check: trend specification, lag length, sample balance.
- When in doubt: prioritize Pedroni (most comprehensive).

**4. Unbalanced panel issues**
- Some tests require balanced panels.
- Solution: Restrict to common time periods.

**5. Assumptions**
- Balanced panel assumed for simplicity.
- No structural breaks (tests assume stable cointegrating relationship).

---

## Exercises

The following exercises allow you to practice applying panel cointegration tests to different datasets. Solutions for Exercises 1-3 are available in `../solutions/02_cointegration_solutions.ipynb`. Exercise 4 is independent (no solution provided).

### Exercise 1: Interest Rate Parity (Easy)

**Task:** Test covered interest rate parity using international interest rates.

**Theory:**
$$i_t - i^*_t = f_t - s_t$$

where $i_t$ = domestic rate, $i^*_t$ = US rate, and $f_t - s_t$ = forward premium.

**Data:** `data/cointegration/interest_rates.csv`  
**Columns:** country, year, domestic_rate, us_rate, spread, forward_premium

**Steps:**
1. Load the interest rate data
2. Verify that `spread` and `forward_premium` are I(1)
3. Test cointegration between `spread` and `forward_premium` using Pedroni
4. Interpret: Does IRP hold in the long run?

In [ ]:
# Exercise 1: Interest Rate Parity
# =================================

# Step 1: Load data
# ir_data = load_dataset("interest_rates", category="diagnostics")
# print(ir_data.head())

# Step 2: Verify variables are I(1)
# YOUR CODE HERE

# Step 3: Test cointegration using Pedroni
# YOUR CODE HERE

# Step 4: Interpret results
# YOUR CODE HERE

### Exercise 2: Compare Methods (Medium)

**Task:** Apply all three methods (Pedroni, Westerlund, Kao) to the OECD consumption-income data.

**Questions:**
1. Do all three methods agree on cointegration?
2. Which method rejects most strongly?
3. How do the results change with different trend specifications (`'c'` vs `'ct'`)?

In [ ]:
# Exercise 2: Compare Methods
# ============================

# Step 1: Run with trend='c'
# YOUR CODE HERE

# Step 2: Run with trend='ct'
# YOUR CODE HERE

# Step 3: Compare and discuss
# YOUR CODE HERE

### Exercise 3: Heterogeneity Analysis (Hard)

**Task:** Investigate heterogeneity in cointegration vectors using the OECD data.

**Steps:**
1. Estimate OLS cointegrating regression $\log(C) = \alpha_i + \beta_i \log(Y) + \varepsilon_{it}$ for each country
2. Extract $\beta_i$ for each country
3. Create two groups: high-MPC ($\beta_i >$ median) and low-MPC ($\beta_i <$ median)
4. Test if the average $\beta$ differs between groups (t-test)
5. Visualize with a bar chart of country-specific $\beta_i$ values

In [ ]:
# Exercise 3: Heterogeneity Analysis
# ====================================

# Step 1: Estimate OLS per country
# YOUR CODE HERE

# Step 2: Extract betas
# YOUR CODE HERE

# Step 3: Group and test differences
# YOUR CODE HERE

# Step 4: Visualize
# YOUR CODE HERE

### Exercise 4: Real Data Application (Independent -- No Solution)

**Task:** Test for cointegration in your own economic relationship.

**Suggestions:**
- Money demand: $M \sim Y, i$
- Trade balance: $TB \sim REER, Y, Y^*$
- Housing prices: $P_{house} \sim Income, Interest\_rate$

**Deliverables:**
1. Theoretical motivation
2. Unit root tests
3. Cointegration tests (at least 2 methods)
4. Interpretation and economic implications

In [ ]:
# Exercise 4: Real Data Application (Independent - No Solution)
# ==============================================================

# Step 1: Choose your economic relationship and justify
# YOUR CODE HERE

# Step 2: Unit root tests
# YOUR CODE HERE

# Step 3: Cointegration tests (at least 2 methods)
# YOUR CODE HERE

# Step 4: Interpretation
# YOUR CODE HERE

---

## Next Steps

Now that you can test for cointegration in panel data, proceed to:

- **Notebook 03: Cross-Sectional Dependence** -- Testing and correcting for dependence across entities
- **Notebook 04: Serial Correlation** -- Diagnosing autocorrelation in panel residuals
- **Notebook 05: Heteroskedasticity** -- Testing for non-constant variance

---

## References

### Seminal Papers

1. **Engle, R. F., & Granger, C. W. J. (1987)**. "Co-integration and error correction: representation, estimation, and testing". *Econometrica*, 55(2), 251-276.
2. **Pedroni, P. (1999)**. "Critical values for cointegration tests in heterogeneous panels with multiple regressors". *Oxford Bulletin of Economics and Statistics*, 61(S1), 653-670.
3. **Pedroni, P. (2004)**. "Panel cointegration: asymptotic and finite sample properties of pooled time series tests with an application to the PPP hypothesis". *Econometric Theory*, 20(3), 597-625.
4. **Westerlund, J. (2007)**. "Testing for error correction in panel data". *Oxford Bulletin of Economics and Statistics*, 69(6), 709-748.
5. **Kao, C. (1999)**. "Spurious regression and residual-based tests for cointegration in panel data". *Journal of Econometrics*, 90(1), 1-44.

### Textbooks

- **Baltagi, B. H. (2021)**. *Econometric Analysis of Panel Data* (6th ed.). Springer. Chapter 12.
- **Banerjee, A., & Wagner, M. (2009)**. "Panel Methods to Test for Unit Roots and Cointegration". In *The Oxford Handbook of Economic Forecasting*.

### Applications

- **MacDonald, R., & Taylor, M. P. (1994)**. "The monetary model of the exchange rate: long-run relationships, short-run dynamics and how to beat a random walk". *Journal of International Money and Finance*, 13(3), 276-290.